[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pythonanywhere/pypath/blob/main/notebooks/module6/09-segmentation.ipynb)

# Module 6.9 — Image Segmentation
**Module 6: Computer Vision** | Estimated time: 30 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Distinguish semantic segmentation from instance segmentation
- Explain FCN and DeepLab architectures at a high level
- Use the Segment Anything Model (SAM) for automatic mask generation
- Walk through the U-Net encoder-decoder architecture with skip connections
- Implement a minimal U-Net in PyTorch
- Remove image backgrounds with the `rembg` library

In [ ]:
!pip install segment-anything --quiet
!pip install rembg --quiet
!pip install opencv-python-headless --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import requests, os
from pathlib import Path

print(f'PyTorch  : {torch.__version__}')
print(f'OpenCV   : {cv2.__version__}')

os.makedirs('/tmp/cv_seg', exist_ok=True)

def show(img, title='', bgr=False, figsize=(8,6)):
    plt.figure(figsize=figsize)
    if bgr:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cmap = 'gray' if len(np.array(img).shape) == 2 else None
    plt.imshow(img, cmap=cmap)
    plt.title(title, fontsize=11); plt.axis('off')
    plt.tight_layout(); plt.show()

# Download a sample image for segmentation
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg'
r = requests.get(url)
with open('/tmp/cv_seg/sample.jpg', 'wb') as f:
    f.write(r.content)
print('Sample image saved.')

img_bgr = cv2.imread('/tmp/cv_seg/sample.jpg')
if img_bgr is None:
    img_bgr = np.random.randint(80, 180, (300, 400, 3), dtype=np.uint8)
    cv2.ellipse(img_bgr, (200, 150), (80, 60), 0, 0, 360, (160, 120, 80), -1)  # cat body
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
print(f'Image shape: {img_bgr.shape}')

## Semantic vs Instance Segmentation

**Semantic segmentation** assigns a class label to every pixel. All instances of the same class share the same colour mask — you know *what* is in each pixel but not *which* object it belongs to.

**Instance segmentation** both detects and delineates individual objects. Each instance gets a unique mask — you know *what* and *which specific object*.

| Property | Semantic | Instance |
|---|---|---|
| All cats same colour? | Yes | No |
| Counts objects? | No | Yes |
| Example models | FCN, DeepLab | Mask R-CNN, YOLOv8-seg |
| Background labelled? | Yes | Sometimes |

In [ ]:
# Visual comparison with a synthetic scene
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Create a synthetic scene with two cats and a dog
scene = np.ones((300, 500, 3), dtype=np.uint8) * 200   # grey background
# Cat 1 (orange)
for pts in [np.array([[100,80],[160,80],[180,200],[80,200]], np.int32)]:
    cv2.fillPoly(scene, [pts], (80, 140, 220))
cv2.circle(scene, (130, 70), 25, (80, 140, 220), -1)   # head
# Cat 2 (orange too, different position)
for pts in [np.array([[280,100],[340,100],[360,220],[260,220]], np.int32)]:
    cv2.fillPoly(scene, [pts], (80, 140, 220))
cv2.circle(scene, (310, 90), 22, (80, 140, 220), -1)
# Dog (brown)
for pts in [np.array([[380,120],[460,120],[470,240],[370,240]], np.int32)]:
    cv2.fillPoly(scene, [pts], (50, 80, 160))
cv2.circle(scene, (415, 110), 28, (50, 80, 160), -1)

# Semantic: all cats one colour, dog another
semantic = np.ones((300, 500, 3), dtype=np.uint8) * 50   # dark background
for pts in [np.array([[100,80],[160,80],[180,200],[80,200]], np.int32),
            np.array([[280,100],[340,100],[360,220],[260,220]], np.int32)]:
    cv2.fillPoly(semantic, [pts], (255, 180, 0))   # all cats → orange
cv2.circle(semantic, (130, 70), 25, (255, 180, 0), -1)
cv2.circle(semantic, (310, 90), 22, (255, 180, 0), -1)
for pts in [np.array([[380,120],[460,120],[470,240],[370,240]], np.int32)]:
    cv2.fillPoly(semantic, [pts], (0, 180, 255))   # dog → blue
cv2.circle(semantic, (415, 110), 28, (0, 180, 255), -1)

# Instance: each object unique colour
instance = np.ones((300, 500, 3), dtype=np.uint8) * 50
colors_inst = [(255, 80, 80), (80, 255, 80), (80, 80, 255)]
for pts, col in zip(
    [np.array([[100,80],[160,80],[180,200],[80,200]], np.int32),
     np.array([[280,100],[340,100],[360,220],[260,220]], np.int32),
     np.array([[380,120],[460,120],[470,240],[370,240]], np.int32)],
    colors_inst):
    cv2.fillPoly(instance, [pts], col)
for center, col in [((130,70),(255,80,80)),((310,90),(80,255,80)),((415,110),(80,80,255))]:
    cv2.circle(instance, center, 22, col, -1)

for ax, img, title in zip(axes,
    [scene, semantic, instance],
    ['Original Scene', 'Semantic Segmentation\n(cat=orange, dog=blue)',
     'Instance Segmentation\n(cat1=red, cat2=green, dog=blue)']):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')

# Legend
patch_sem  = [mpatches.Patch(color=(1,.7,0), label='Cat (class)'),
              mpatches.Patch(color=(0,.7,1), label='Dog (class)')]
patch_inst = [mpatches.Patch(color=(1,.3,.3), label='Cat 1'),
              mpatches.Patch(color=(.3,1,.3), label='Cat 2'),
              mpatches.Patch(color=(.3,.3,1), label='Dog')]
axes[1].legend(handles=patch_sem,  loc='lower right', fontsize=8)
axes[2].legend(handles=patch_inst, loc='lower right', fontsize=8)

plt.suptitle('Semantic vs Instance Segmentation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## FCN and DeepLab Overview

**FCN (Fully Convolutional Network)** — the foundational semantic segmentation model (2015):
- Replaces FC layers with convolutional layers → can accept any input size
- Upsamples via transposed convolutions to produce a per-pixel output
- Uses skip connections from early layers to recover spatial detail

**DeepLab** — Google's semantic segmentation series:
- Uses **atrous (dilated) convolutions** to increase receptive field without losing resolution
- **ASPP** (Atrous Spatial Pyramid Pooling): parallel dilated convolutions at multiple rates
- DeepLabv3+ adds a decoder with skip connections for sharper boundaries

In [ ]:
# Visualise dilated convolution concept
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, rate, title in zip(axes, [1, 2, 4],
    ['Standard Conv\n(dilation rate=1)', 'Dilated Conv\n(rate=2)',
     'Dilated Conv\n(rate=4)']):
    grid = np.ones((9, 9, 3), dtype=np.float32) * 0.9
    cx, cy = 4, 4   # centre
    # Draw the receptive field points
    for i in range(3):
        for j in range(3):
            px = cx + (i-1) * rate
            py = cy + (j-1) * rate
            if 0 <= px < 9 and 0 <= py < 9:
                grid[py, px] = [0.2, 0.6, 1.0]   # blue = sampled
    grid[cy, cx] = [1.0, 0.3, 0.3]               # red = centre
    ax.imshow(grid, interpolation='nearest')
    ax.set_title(title, fontsize=9)
    # Draw grid lines
    for k in range(10):
        ax.axhline(k - 0.5, color='gray', linewidth=0.5)
        ax.axvline(k - 0.5, color='gray', linewidth=0.5)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Dilated (Atrous) Convolution — Same parameters, larger receptive field',
             fontweight='bold')
plt.tight_layout(); plt.show()

print('Same 3×3 kernel, different dilation rates:')
print('  rate=1 → 3×3 receptive field  (standard)')
print('  rate=2 → 5×5 receptive field  (skips 1 pixel)')
print('  rate=4 → 9×9 receptive field  (skips 3 pixels)')

## Segment Anything Model (SAM)

Meta's SAM (2023) can segment any object from a prompt (point, box, or nothing). The automatic mask generator finds all plausible masks in an image.

**Note:** The SAM ViT-B checkpoint is ~375 MB. We download it below. If bandwidth is limited, the code handles the fallback gracefully.

In [ ]:
import urllib.request

SAM_CHECKPOINT = '/tmp/cv_seg/sam_vit_b_01ec64.pth'
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

if not os.path.exists(SAM_CHECKPOINT):
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    print('This may take a few minutes on Colab.')
    try:
        urllib.request.urlretrieve(SAM_URL, SAM_CHECKPOINT)
        print(f'Downloaded: {os.path.getsize(SAM_CHECKPOINT)//1024//1024} MB')
    except Exception as e:
        print(f'Download failed: {e}')
        print('Proceeding without SAM — using synthetic segmentation demo.')
else:
    print(f'SAM checkpoint found: {os.path.getsize(SAM_CHECKPOINT)//1024//1024} MB')

In [ ]:
def run_sam_automatic(image_rgb, checkpoint_path, device='cpu'):
    """Run SAM automatic mask generation."""
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    sam = sam_model_registry['vit_b'](checkpoint=checkpoint_path)
    sam.to(device=device)
    mask_gen = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=16,           # fewer = faster
        pred_iou_thresh=0.88,
        stability_score_thresh=0.95,
        min_mask_region_area=200,
    )
    masks = mask_gen.generate(image_rgb)
    return masks

def show_sam_masks(image_rgb, masks):
    """Overlay SAM masks with random colours."""
    overlay = image_rgb.copy().astype(np.float32)
    np.random.seed(42)
    for mask_data in masks:
        m = mask_data['segmentation']
        colour = np.random.randint(50, 255, 3).astype(float)
        overlay[m] = overlay[m] * 0.5 + colour * 0.5
    overlay = overlay.clip(0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(image_rgb); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(overlay)
    axes[1].set_title(f'SAM — {len(masks)} masks detected'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

# Run SAM if checkpoint exists
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.exists(SAM_CHECKPOINT) and os.path.getsize(SAM_CHECKPOINT) > 1_000_000:
    try:
        print(f'Running SAM on {DEVICE}...')
        masks = run_sam_automatic(img_rgb, SAM_CHECKPOINT, DEVICE)
        print(f'Masks generated: {len(masks)}')
        if masks:
            # Sort by area, show properties
            masks.sort(key=lambda x: x['area'], reverse=True)
            print('\nTop-5 masks by area:')
            for i, m in enumerate(masks[:5]):
                print(f'  Mask {i+1}: area={m["area"]:,}px  '
                      f'confidence={m["predicted_iou"]:.3f}')
            show_sam_masks(img_rgb, masks)
    except Exception as e:
        print(f'SAM error: {e}. Using fallback demonstration.')
        masks = []
else:
    print('SAM checkpoint not available — showing synthetic mask demo.')
    # Synthetic demo using colour-based segmentation
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    # Segment roughly by hue
    n_segments = 5
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_rgb); axes[0].set_title('Original'); axes[0].axis('off')

    # Simple k-means colour segmentation as SAM stand-in
    pixels = img_rgb.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(pixels, n_segments, None, criteria, 5,
                                    cv2.KMEANS_RANDOM_CENTERS)
    segmented = centers[labels.flatten()].reshape(img_rgb.shape).astype(np.uint8)
    axes[1].imshow(segmented)
    axes[1].set_title(f'K-Means ({n_segments} segments) — SAM placeholder')
    axes[1].axis('off')
    plt.suptitle('Segmentation Demo (run on Colab for full SAM)', fontweight='bold')
    plt.tight_layout(); plt.show()

## U-Net Architecture

U-Net (Ronneberger et al., 2015) is the dominant architecture for medical and satellite image segmentation. It uses an **encoder-decoder** structure with **skip connections**:

```
Encoder (contracting path)          Decoder (expanding path)
  Input (1×572×572)                   Output (1×388×388)
  Conv→ReLU×2                              ↑
  MaxPool ─────────────────────────→ Cat + UpConv
  Conv→ReLU×2                              ↑
  MaxPool ─────────────────────────→ Cat + UpConv
  Conv→ReLU×2                              ↑
  MaxPool ─────────────────────────→ Cat + UpConv
  Bottleneck (Conv→ReLU×2)
```

The skip connections copy feature maps from the encoder directly to the decoder at the same resolution level. This restores spatial detail lost during downsampling.

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive Conv-BN-ReLU blocks."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    """
    Simplified U-Net for binary segmentation.
    Input:  (B, in_channels, H, W)
    Output: (B, num_classes, H, W)
    """
    def __init__(self, in_channels=3, num_classes=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoder   = nn.ModuleList()
        self.decoder   = nn.ModuleList()
        self.pool      = nn.MaxPool2d(2, 2)

        # Encoder
        ch = in_channels
        for f in features:
            self.encoder.append(DoubleConv(ch, f))
            ch = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder
        for f in reversed(features):
            self.decoder.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.decoder.append(DoubleConv(f * 2, f))  # f*2 because of skip

        # Final 1×1 conv
        self.final = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skip_connections = []

        # Encoder path
        for enc in self.encoder:
            x = enc(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]  # reverse for decoder

        # Decoder path
        for i in range(0, len(self.decoder), 2):
            x     = self.decoder[i](x)          # upsampling
            skip  = skip_connections[i // 2]    # corresponding encoder feature
            # Handle size mismatch
            if x.shape != skip.shape:
                x = nn.functional.interpolate(
                    x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([skip, x], dim=1)     # skip connection
            x = self.decoder[i + 1](x)          # double conv

        return self.final(x)

unet = UNet(in_channels=3, num_classes=1, features=[32, 64, 128, 256])
print('U-Net architecture:')
print(unet)

# Count parameters
total = sum(p.numel() for p in unet.parameters())
print(f'\nTotal parameters: {total:,}')

# Test forward pass
with torch.no_grad():
    x_test = torch.randn(2, 3, 256, 256)
    y_test = unet(x_test)
print(f'Input:  {x_test.shape}')
print(f'Output: {y_test.shape}  (logits — apply sigmoid for binary mask)')

## U-Net Architecture Visualisation

In [ ]:
# Visualise U-Net architecture as a diagram
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 8); ax.axis('off')

stages = [
    # (x, y, w, h, label, colour)
    (0.2, 5.5, 1.2, 2.0, 'Input\n3×256×256', '#AED6F1'),
    (2.0, 5.5, 1.2, 2.0, 'Enc1\n32×256', '#82E0AA'),
    (3.8, 4.5, 1.2, 1.8, 'Enc2\n64×128', '#82E0AA'),
    (5.6, 3.5, 1.2, 1.6, 'Enc3\n128×64', '#82E0AA'),
    (7.4, 2.5, 1.2, 1.4, 'Bottleneck\n512×32', '#F8C471'),
    (9.2, 3.5, 1.2, 1.6, 'Dec3\n128×64', '#F1948A'),
    (11.0, 4.5, 1.2, 1.8, 'Dec2\n64×128', '#F1948A'),
    (12.8, 5.5, 0.8, 2.0, 'Out\n1×256', '#D2B4DE'),
]

for x, y, w, h, label, col in stages:
    rect = plt.Rectangle((x, y), w, h, linewidth=2,
                          edgecolor='#555', facecolor=col, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=8, fontweight='bold')

# Arrows (encoder path + bottleneck + decoder)
for src, dst, skip in [
    ((1.4,6.5),(2.0,6.5), False),
    ((2.6,5.5),(3.8,5.5), False),  # pool
    ((4.4,4.5),(5.6,4.5), False),
    ((6.2,3.5),(7.4,3.5), False),
    ((8.6,3.2),(9.2,3.8), False),  # up
    ((10.4,4.5),(11.0,5.0),False),
    ((12.0,5.5),(12.8,6.0),False),
]:
    ax.annotate('', xy=dst, xytext=src,
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Skip connections
for (x1,y1),(x2,y2),label in [
    ((2.6,7.0),(11.0+0.6,7.0), 'skip 1'),
    ((4.4,6.0),(9.2+0.6,6.0),  'skip 2'),
    ((6.2,5.0),(9.2-0.8,5.0),  'skip 3'),
]:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='steelblue',
                                lw=1.5, linestyle='dashed'))
    ax.text((x1+x2)/2, y1+0.2, label, ha='center', fontsize=7, color='steelblue')

ax.set_title('U-Net Architecture — Encoder-Decoder with Skip Connections',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Background Removal with rembg

`rembg` is a lightweight library that uses a pre-trained deep learning model (U²-Net) to automatically remove backgrounds from images. It is widely used for product photography and portrait editing.

In [ ]:
from rembg import remove
from PIL import Image as PILImage
import io

# Load sample image
with open('/tmp/cv_seg/sample.jpg', 'rb') as f:
    input_bytes = f.read()

print('Running background removal (first run downloads U²-Net weights ~170 MB)...')
try:
    output_bytes = remove(input_bytes)
    out_img = PILImage.open(io.BytesIO(output_bytes)).convert('RGBA')
    # Save as PNG (preserves transparency)
    out_img.save('/tmp/cv_seg/no_bg.png')
    print(f'Background removed. Output size: {out_img.size}')

    # Visualise
    orig_img = PILImage.open('/tmp/cv_seg/sample.jpg').convert('RGB')

    # Composite on white and green backgrounds
    white_bg = PILImage.new('RGB', out_img.size, (255, 255, 255))
    white_bg.paste(out_img, mask=out_img.split()[3])
    green_bg = PILImage.new('RGB', out_img.size, (50, 200, 100))
    green_bg.paste(out_img, mask=out_img.split()[3])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(white_bg); axes[1].set_title('White Background'); axes[1].axis('off')
    axes[2].imshow(green_bg); axes[2].set_title('Green Background'); axes[2].axis('off')
    plt.suptitle('rembg Background Removal', fontweight='bold')
    plt.tight_layout(); plt.show()

except Exception as e:
    print(f'rembg error: {e}')
    print('Demonstrating manual background removal with OpenCV GrabCut instead.')

    # GrabCut fallback
    img = img_bgr.copy()
    mask    = np.zeros(img.shape[:2], np.uint8)
    bgd_mdl = np.zeros((1, 65), np.float64)
    fgd_mdl = np.zeros((1, 65), np.float64)

    h, w = img.shape[:2]
    rect = (20, 20, w - 40, h - 40)   # initial foreground rect
    cv2.grabCut(img, mask, rect, bgd_mdl, fgd_mdl, 5, cv2.GC_INIT_WITH_RECT)

    fg_mask = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
    result  = img * fg_mask[:, :, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    axes[1].set_title('GrabCut Background Removal'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

## Summary

| Technique | Type | Model | Strength |
|---|---|---|---|
| FCN | Semantic | Fully conv. VGG | Fast, pioneering |
| DeepLab | Semantic | Dilated convs + ASPP | High accuracy |
| SAM | Instance (any) | ViT + MAE | Zero-shot, universal |
| U-Net | Semantic / binary | Encoder-decoder | Medical images, small data |
| Mask R-CNN | Instance | FPN + RoI | Standard benchmark |
| rembg / U²-Net | Binary | U²-Net | Background removal |

## Practice Exercises

**Exercise 1 — U-Net Training:**  
Create a synthetic segmentation dataset: generate 200 images with random filled circles on noisy backgrounds. The mask should be 1 inside circles, 0 outside. Train the `UNet` with `BCEWithLogitsLoss`. Plot loss curves and visualise predicted masks vs. ground truth after training.

**Exercise 2 — SAM Point Prompt:**  
Instead of automatic mask generation, use SAM with a point prompt (`SamPredictor`). Load an image, click on an object (specify x,y coordinates), and generate a mask. Compare the point-prompted mask to the automatic mask for the same object.

**Exercise 3 — Background Replacement:**  
Use `rembg` to remove the background from a portrait photo. Replace it with three different backgrounds: a beach scene, a blur of the original background (`cv2.GaussianBlur`), and a solid colour. Save all three versions and display them in a 2×2 grid.